# 03 - 误差分析（基于预测结果）

读取推理生成的 `predictions.csv`，统计 TP/FP/FN/TN，并导出漏报（FN）、误报（FP）清单。

**路径解析**：与 CLI 一致，优先 `runs/<run_name>/.latest_infer_session`，否则根目录 `predictions.csv`。

**自动补齐链路**：第二格先查是否已有 **`predictions.csv`**（有则直接分析）。否则检查并补齐 **`data/splits`**、**checkpoint**、推理产物；缺则自动执行（与终端一致）：`preprocess.sh` → `train.sh` → `infer.sh`（`EXP_YAML` 见首格）。若 splits 已存在但 **`X_train` 的 `(C,H,W)` 与 `configs/model.yaml` 不一致**（例如改过网格尺寸），会**自动重跑 preprocess**，避免训练阶段形状错误。

**关于旧报错**：若单元格输出里仍出现 **`window_days` / `validate_config`** 相关 `ValueError`，那是 Jupyter 缓存的旧输出；当前仓库校验只使用 **`data.window_seconds`**。请「清除输出」后重跑。若自动训练后提示 **找不到 checkpoint**，常见原因是验证集 **PR-AUC（avg_precision）全为 nan** 导致旧版从未 `torch.save`；当前训练器已修复（首轮可落盘 + 末轮兜底）。

**输出**：`outputs/tables/error_confusion_counts.csv`、`error_FN_samples.csv`、`error_FP_samples.csv`


In [1]:
from pathlib import Path
import sys
# 自动判断项目根目录
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
_SRC = str(PROJECT_ROOT / "src")
# 把 src/ 目录加入 Python 模块搜索路径（优先从 src 找模块）
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

# Jupyter 会话可能已加载过旧版 dram2cnn（如 site-packages），导致仍报已删除的 window_days 等校验
# 强制清除 Jupyter Notebook / IPython 会话中已经加载过的旧版 dram2cnn 模块。
# Jupyter 会把导入过的模块缓存在 sys.modules 里，即使你修改了源码、删除了某些函数（比如 window_days），它仍然会使用缓存的旧版本，导致报错。
# 这段代码就是把所有以 dram2cnn 开头的模块全部从内存中删除，下次 import dram2cnn.xxx 时就会重新加载最新的源码。
# 以下清理 Jupyter 中缓存的旧版 dram2cnn 模块
for _name in list(sys.modules):
    if _name == "dram2cnn" or _name.startswith("dram2cnn."):
        del sys.modules[_name]

# 设置实验配置文件路径
# 与当前分析一致的实验配置（与 infer 时使用的 yaml 保持一致）
EXP_YAML = PROJECT_ROOT / "configs" / "experiments" / "exp_baseline.yaml"
# 创建输出表格目录（不存在就自动创建）
OUT_T = PROJECT_ROOT / "outputs" / "tables"
OUT_T.mkdir(parents=True, exist_ok=True)


In [2]:
import os
import subprocess
from pathlib import Path

import numpy as np

from dram2cnn.utils.config import load_experiment_config
from dram2cnn.utils.run_layout import read_pointer_dir, resolve_checkpoint_dir

try:
    _EXP_REL_STR = str(EXP_YAML.relative_to(PROJECT_ROOT))
except ValueError:
    _EXP_REL_STR = str(EXP_YAML.resolve())


def _run_shell_script(rel_script: str, arg: str) -> None:
    cmd = ["bash", str(PROJECT_ROOT / rel_script), arg]
    env = os.environ.copy()
    src_abs = str(PROJECT_ROOT / "src")
    pp = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = f"{src_abs}:{pp}" if pp else src_abs
    print("提示：执行", " ".join(cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, check=True)


exp_path = EXP_YAML.resolve()
from dram2cnn.utils.preprocess_preflight import experiment_run_name

run_name = experiment_run_name(exp_path)
run_root = PROJECT_ROOT / "runs" / run_name
run_root.mkdir(parents=True, exist_ok=True)

infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
pred_path = None
if infer_dir is not None:
    cand = infer_dir / "predictions.csv"
    if cand.is_file():
        pred_path = cand
if pred_path is None and (run_root / "predictions.csv").is_file():
    infer_dir = run_root
    pred_path = infer_dir / "predictions.csv"

cfg = None
if pred_path is not None and pred_path.is_file():
    print("提示：已找到 predictions.csv，跳过预处理、训练与推理（无需加载完整实验配置）。")
else:
    try:
        cfg = load_experiment_config(str(exp_path))
    except FileNotFoundError as e:
        raise FileNotFoundError(
            "加载实验配置失败（常见原因：data.yaml 中的 mcelog/tickets/index_features 等路径不存在）。\n"
            f"原始错误: {e}\n"
            "若仅需误差分析，请先将 predictions.csv 放到 runs/<run_name>/.latest_infer_session/ 下。"
        ) from e
    run_name = str(cfg["base"].get("run_name", run_name))
    run_root = PROJECT_ROOT / "runs" / run_name
    run_root.mkdir(parents=True, exist_ok=True)

    splits_dir = Path(cfg["data"]["splits_dir"])
    if not splits_dir.is_absolute():
        splits_dir = PROJECT_ROOT / splits_dir

    def _split_x_shape_matches_model(sd: Path, model_cfg: dict) -> bool:
        xt = sd / "X_train.npy"
        if not xt.is_file():
            return False
        x = np.load(xt, mmap_mode="r")
        if x.ndim != 4:
            return False
        exp_c = int(model_cfg.get("in_channels", x.shape[1]))
        exp_h = int(model_cfg.get("grid_height", x.shape[2]))
        exp_w = int(model_cfg.get("grid_width", x.shape[3]))
        _n, c, h, w = x.shape
        return int(c) == exp_c and int(h) == exp_h and int(w) == exp_w

    def _splits_ready(sd: Path) -> bool:
        td = int(cfg["model"].get("tabular_dim", 0))
        for tag in ("train", "val", "test"):
            if not (sd / f"X_{tag}.npy").is_file():
                return False
            if not (sd / f"y_{tag}.npy").is_file():
                return False
            if td > 0 and not (sd / f"tabular_{tag}.npy").is_file():
                return False
        return (sd / "normalization_stats.json").is_file()

    if not _splits_ready(splits_dir):
        print("提示：data/splits 不完整，自动运行预处理（耗时取决于原始数据规模）。")
        _run_shell_script("scripts/preprocess.sh", _EXP_REL_STR)
    elif not _split_x_shape_matches_model(splits_dir, cfg["model"]):
        x = np.load(splits_dir / "X_train.npy", mmap_mode="r")
        chw = tuple(int(x.shape[i]) for i in range(1, 4)) if x.ndim == 4 else x.shape
        mc = cfg["model"]
        exp = (int(mc.get("in_channels", 3)), int(mc.get("grid_height", 0)), int(mc.get("grid_width", 0)))
        print(
            f"提示：现有切分 X_train (C,H,W)={chw} 与 model.yaml 期望 {exp} 不一致，将重新运行 preprocess。"
        )
        _run_shell_script("scripts/preprocess.sh", _EXP_REL_STR)

    if not _splits_ready(splits_dir):
        raise RuntimeError(f"预处理后仍缺少 splits，请检查 data/raw 与配置：{splits_dir}")
    if not _split_x_shape_matches_model(splits_dir, cfg["model"]):
        raise RuntimeError(
            f"预处理后 X_train 形状仍与 model.yaml 不一致：{splits_dir}。"
            "请检查 data.grid_max_* / model.grid_height / model.grid_width 并重跑 preprocess。"
        )

    try:
        resolve_checkpoint_dir(PROJECT_ROOT, cfg["base"])
    except FileNotFoundError:
        print("提示：未找到 checkpoint，自动运行训练（可能非常久；与 scripts/train.sh 一致，含 DDP 逻辑）。")
        _run_shell_script("scripts/train.sh", _EXP_REL_STR)

    try:
        resolve_checkpoint_dir(PROJECT_ROOT, cfg["base"])
    except FileNotFoundError as exc:
        raise RuntimeError(f"训练结束后仍未解析到 checkpoint：{exc}") from exc

    infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
    pred_path = None
    if infer_dir is not None:
        cand = infer_dir / "predictions.csv"
        if cand.is_file():
            pred_path = cand
    if pred_path is None and (run_root / "predictions.csv").is_file():
        infer_dir = run_root
        pred_path = infer_dir / "predictions.csv"

    if pred_path is None or not pred_path.is_file():
        print("提示：未找到 predictions.csv，自动运行推理。")
        _run_shell_script("scripts/infer.sh", _EXP_REL_STR)
        infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
        pred_path = None
        if infer_dir is not None:
            cand = infer_dir / "predictions.csv"
            if cand.is_file():
                pred_path = cand
        if pred_path is None and (run_root / "predictions.csv").is_file():
            infer_dir = run_root
            pred_path = infer_dir / "predictions.csv"

    if pred_path is None or not pred_path.is_file():
        raise RuntimeError(f"推理后仍找不到 predictions.csv，请检查 {run_root}")

infer_dir = pred_path.parent
print("run_name:", run_name)
print("infer_dir:", infer_dir)
print("predictions:", pred_path)


提示：未找到 checkpoint，自动运行训练（可能非常久；与 scripts/train.sh 一致，含 DDP 逻辑）。
提示：执行 bash /root/workspace/scripts/train.sh configs/experiments/exp_baseline.yaml
[train.sh] device configured=auto, resolved=cuda, cuda_count=3, ddp_enabled=1, nproc_per_node=3
[train.sh] launch_plan using_all_visible_gpus=1 (1=yes,0=no)
[train.sh] DDP enabled: torchrun --nproc_per_node=3 -m dram2cnn.pipelines.run_train configs/experiments/exp_baseline.yaml


W0512 14:26:00.997000 847002 torch/distributed/run.py:851] 
W0512 14:26:00.997000 847002 torch/distributed/run.py:851] *****************************************
W0512 14:26:00.997000 847002 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0512 14:26:00.997000 847002 torch/distributed/run.py:851] *****************************************
2026-05-12 14:26:04,348 [INFO] Train launch context: world_size=3, rank=0, local_rank=0, configured_device=auto, resolved_device=cuda, cuda_available=True, cuda_count=3, using_all_visible_gpus=True
/root/workspace/src/dram2cnn/data/dataset.py:52: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data

提示：未找到 predictions.csv，自动运行推理。
提示：执行 bash /root/workspace/scripts/infer.sh configs/experiments/exp_baseline.yaml


/root/workspace/src/dram2cnn/data/dataset.py:52: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /opt/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.x = torch.from_numpy(x_np)


Inference done: checkpoint_dir=/root/workspace/runs/exp_tianchi_baseline/20260512_142604_245046
  predictions: /root/workspace/runs/exp_tianchi_baseline/infer_20260512_142702_713488/predictions.csv
run_name: exp_tianchi_baseline
infer_dir: /root/workspace/runs/exp_tianchi_baseline/infer_20260512_142702_713488
predictions: /root/workspace/runs/exp_tianchi_baseline/infer_20260512_142702_713488/predictions.csv


In [3]:
import pandas as pd

if "pred_path" not in globals() or pred_path is None:
    raise RuntimeError("请先自上而下运行上一单元（须成功解析 predictions）。")

df = pd.read_csv(pred_path)
need = {"y_true", "y_pred"}
if not need.issubset(df.columns):
    raise ValueError(f"predictions.csv 需含列 {need}，实际: {list(df.columns)}")

yt = df["y_true"].astype(int)
yp = df["y_pred"].astype(int)

tp = int(((yt == 1) & (yp == 1)).sum())
fp = int(((yt == 0) & (yp == 1)).sum())
fn = int(((yt == 1) & (yp == 0)).sum())
tn = int(((yt == 0) & (yp == 0)).sum())

summary = pd.DataFrame([
    {"name": "TP", "count": tp},
    {"name": "FP", "count": fp},
    {"name": "FN", "count": fn},
    {"name": "TN", "count": tn},
])
summary.to_csv(OUT_T / "error_confusion_counts.csv", index=False)
print(summary)

fn_df = df[(yt == 1) & (yp == 0)].copy()
fp_df = df[(yt == 0) & (yp == 1)].copy()
fn_df.to_csv(OUT_T / "error_FN_samples.csv", index=False)
fp_df.to_csv(OUT_T / "error_FP_samples.csv", index=False)
print("FN rows", len(fn_df), "FP rows", len(fp_df))
print("已写入 outputs/tables/error_*.csv")


  name  count
0   TP      1
1   FP      0
2   FN      0
3   TN      8
FN rows 0 FP rows 0
已写入 outputs/tables/error_*.csv
